# Experiment 2 Postprocessing

This notebook generates saved artifacts for the silo-mode comparison experiment.

Required input map:
- `experiments/exp2_parallelism_ablation/outputs/last_run_manifest.json`


In [6]:
from pathlib import Path
import csv
import json
from typing import Any
import matplotlib.pyplot as plt

plt.rcParams.update({
    "figure.figsize": (6, 4),      # width, height in inches
    "font.size": 14,               # base font size
    "axes.titlesize": 16,
    "axes.labelsize": 15,
    "xtick.labelsize": 13,
    "ytick.labelsize": 13,
    "legend.fontsize": 13,
    "figure.dpi": 300,             # high resolution for paper
})

import pandas as pd

# Pandas compatibility shim: some environments reject errors='ignore' in to_numeric.
if not hasattr(pd, '_flsim_to_numeric_compat'):
    _orig_to_numeric = pd.to_numeric

    def _to_numeric_compat(arg, errors='raise', **kwargs):
        if errors == 'ignore':
            converted = _orig_to_numeric(arg, errors='coerce', **kwargs)
            if hasattr(arg, 'where'):
                return converted.where(converted.notna(), arg)
            return arg
        return _orig_to_numeric(arg, errors=errors, **kwargs)

    pd.to_numeric = _to_numeric_compat
    pd._flsim_to_numeric_compat = True

def find_repo_root(start: Path) -> Path | None:
    for candidate in [start, *start.parents]:
        if (candidate / 'orchestrator.py').exists() and (candidate / 'experiments').is_dir():
            return candidate
    return None
ROOT = find_repo_root(Path.cwd())
if ROOT is None:
    raise RuntimeError(
        f'Could not locate fl-net-sim repo root from current working directory: {Path.cwd()}'
    )
EXP_ID = 'exp2_parallelism_ablation'
EXP_DIR = ROOT / 'experiments' / EXP_ID
MANIFEST_PATH = EXP_DIR / 'outputs' / 'last_run_manifest.json'
TABLES_DIR = EXP_DIR / 'outputs' / 'tables'
FIGURES_DIR = EXP_DIR / 'outputs' / 'figures'
TABLES_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)


In [7]:
if not MANIFEST_PATH.exists():
    raise FileNotFoundError(f'Missing manifest: {MANIFEST_PATH}. Run the experiment first.')
manifest = json.loads(MANIFEST_PATH.read_text(encoding='utf-8'))
runs = manifest.get('runs', [])
if not runs:
    raise ValueError(f'No runs in manifest: {MANIFEST_PATH}')
LABEL_BY_CONFIG_ID = {
    'baseline_ns3': 'Intra-process: Off | Inter-process: Off',
    'unison_auto_p1': 'Intra-process: On | Inter-process: Off',
    'parallelism20_no_unison': 'Intra-process: Off | Inter-process: On',
    'parallelism20_unison_t2': 'Intra-process: On | Inter-process: On',
}
runs_df = pd.DataFrame(runs)
if 'config_label' not in runs_df.columns:
    runs_df['config_label'] = runs_df['config_id']
runs_df['config_label'] = runs_df['config_id'].map(LABEL_BY_CONFIG_ID).fillna(runs_df['config_label'])
runs_df['rounds_requested'] = pd.to_numeric(runs_df.get('rounds_requested', 0), errors='coerce').fillna(0).astype(int)
runs_df['planned_rounds'] = pd.to_numeric(runs_df.get('planned_rounds', 0), errors='coerce').fillna(0).astype(int)
runs_df.loc[runs_df['planned_rounds'] <= 0, 'planned_rounds'] = runs_df.loc[runs_df['planned_rounds'] <= 0, 'rounds_requested']
runs_df[['config_id', 'config_label', 'exec_time_s', 'rounds_requested', 'planned_rounds', 'cached_rounds', 'deduped_rounds', 'parallelism_reported', 'exports_dir']]


,config_id,config_label,exec_time_s,rounds_requested,planned_rounds,cached_rounds,deduped_rounds,parallelism_reported,exports_dir
0,baseline_ns3,Intra-process: Off | Inter-process: Off,668.919876,100,100,0,25,1,/home/psklavos/nas_drive/psklavos/MDM2026/fl-n...
1,unison_auto_p1,Intra-process: On | Inter-process: Off,476.838256,100,100,0,25,1,/home/psklavos/nas_drive/psklavos/MDM2026/fl-n...
2,parallelism20_no_unison,Intra-process: Off | Inter-process: On,79.940318,100,100,0,25,20,/home/psklavos/nas_drive/psklavos/MDM2026/fl-n...
3,parallelism20_unison_t2,Intra-process: On | Inter-process: On,46.354366,100,100,0,25,20,/home/psklavos/nas_drive/psklavos/MDM2026/fl-n...


In [8]:
import re
from datetime import datetime

def parse_runtime_round_table(log_path: Path) -> pd.DataFrame:
    starts: dict[int, datetime] = {}
    rows: list[dict[str, Any]] = []

    ts_patterns = ['%Y-%m-%d %H:%M:%S', '%Y-%m-%d %H:%M:%S,%f']
    line_re = re.compile(r'^(?P<ts>\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2}(?:,\d+)?) \| INFO \| round (?P<round>\d+) (?P<event>start|complete)$')

    for line in log_path.read_text(encoding='utf-8', errors='ignore').splitlines():
        m = line_re.match(line)
        if not m:
            continue

        ts_text = m.group('ts')
        ts = None
        for fmt in ts_patterns:
            try:
                ts = datetime.strptime(ts_text, fmt)
                break
            except ValueError:
                continue
        if ts is None:
            continue

        round_id = int(m.group('round'))
        event = m.group('event')

        if event == 'start':
            starts[round_id] = ts
        elif round_id in starts:
            start_ts = starts[round_id]
            rows.append(
                {
                    'round': round_id,
                    'round_start_ts': start_ts,
                    'round_end_ts': ts,
                    'wall_time_s': (ts - start_ts).total_seconds(),
                }
            )

    frame = pd.DataFrame(rows)
    if frame.empty:
        return frame

    frame['round_start_ts'] = pd.to_datetime(frame['round_start_ts'])
    frame['round_end_ts'] = pd.to_datetime(frame['round_end_ts'])
    run_start = frame['round_start_ts'].min()
    frame['completion_elapsed_s'] = (frame['round_end_ts'] - run_start).dt.total_seconds()
    frame['round_start_wallclock'] = frame['round_start_ts'].dt.strftime('%Y-%m-%d %H:%M:%S')
    frame['round_end_wallclock'] = frame['round_end_ts'].dt.strftime('%Y-%m-%d %H:%M:%S')
    return frame.drop(columns=['round_start_ts', 'round_end_ts'])

round_rows: list[dict[str, Any]] = []
wall_rows: list[pd.DataFrame] = []

for item in runs:
    config_id = item.get('config_id')
    config_label = LABEL_BY_CONFIG_ID.get(config_id, item.get('config_label', config_id))

    log_candidates: list[Path] = []
    exports_dir = item.get('exports_dir')
    if exports_dir:
        log_candidates.append(Path(exports_dir) / 'orchestrator_capture.log')
    log_candidates.append(EXP_DIR / 'outputs' / 'run_logs' / f'{config_id}.log')

    capture_log = next((candidate for candidate in log_candidates if candidate.exists()), None)
    if capture_log is None:
        continue

    frame = parse_runtime_round_table(capture_log)
    if frame.empty:
        continue

    frame['config_id'] = config_id
    frame['config_label'] = config_label
    wall_rows.append(frame)

    wall = pd.to_numeric(frame['wall_time_s'], errors='coerce').dropna()
    completion = pd.to_numeric(frame['completion_elapsed_s'], errors='coerce').dropna()
    if wall.empty:
        continue

    round_rows.append(
        {
            'config_id': config_id,
            'config_label': config_label,
            'mean_round_wall_time_s': float(wall.mean()),
            'p50_round_wall_time_s': float(wall.quantile(0.5)),
            'p95_round_wall_time_s': float(wall.quantile(0.95)),
            'min_round_wall_time_s': float(wall.min()),
            'max_round_wall_time_s': float(wall.max()),
            'std_round_wall_time_s': float(wall.std(ddof=0)),
            'sum_round_wall_time_s': float(wall.sum()),
            'max_completion_elapsed_s': float(completion.max()) if not completion.empty else float('nan'),
        }
    )

per_round_stats = pd.DataFrame(round_rows)
all_wall = pd.concat(wall_rows, ignore_index=True) if wall_rows else pd.DataFrame()

if not per_round_stats.empty:
    print('using orchestrator runtime logs for per-round wall-time statistics.')
    print('note: with inter-process parallelism, per-round durations can increase due to overlap/CPU contention while total execution still improves.')

None


using orchestrator runtime logs for per-round wall-time statistics.
note: with inter-process parallelism, per-round durations can increase due to overlap/CPU contention while total execution still improves.


In [9]:
summary = runs_df[['config_id', 'config_label', 'exec_time_s', 'rounds_requested', 'planned_rounds', 'cached_rounds', 'deduped_rounds', 'parallelism_reported']].copy()
summary['avg_time_per_round_s'] = summary['exec_time_s'] / summary['rounds_requested'].clip(lower=1)
summary['dedup_ratio'] = summary['deduped_rounds'] / summary['rounds_requested'].clip(lower=1)
summary['cached_ratio'] = summary['cached_rounds'] / summary['rounds_requested'].clip(lower=1)

if not per_round_stats.empty:
    stats_cols = [
        'config_id',
        'mean_round_wall_time_s',
        'p50_round_wall_time_s',
        'p95_round_wall_time_s',
        'min_round_wall_time_s',
        'max_round_wall_time_s',
        'std_round_wall_time_s',
        'sum_round_wall_time_s',
        'max_completion_elapsed_s',
    ]
    summary = summary.merge(per_round_stats[stats_cols], on='config_id', how='left')

baseline_rows = summary[summary['config_id'] == 'baseline_ns3']
baseline_exec = float(baseline_rows['exec_time_s'].iloc[0]) if not baseline_rows.empty else None
summary['speedup_vs_baseline'] = baseline_exec / summary['exec_time_s'] if baseline_exec else None

if 'sum_round_wall_time_s' in summary.columns:
    summary['overlap_factor'] = summary['sum_round_wall_time_s'] / summary['exec_time_s'].replace(0, pd.NA)
else:
    summary['overlap_factor'] = pd.NA

summary = summary.sort_values('config_id').reset_index(drop=True)

def to_markdown_no_deps(df: pd.DataFrame) -> str:
    try:
        return df.to_markdown(index=False)
    except Exception:
        safe = df.fillna('')
        cols = [str(c) for c in safe.columns]
        header = '| ' + ' | '.join(cols) + ' |'
        sep = '| ' + ' | '.join(['---'] * len(cols)) + ' |'
        rows = [
            '| ' + ' | '.join(str(v).replace('\n', ' ').replace('|', '\\|') for v in row) + ' |'
            for row in safe.itertuples(index=False, name=None)
        ]
        return '\n'.join([header, sep, *rows])

summary_core = summary[['config_id', 'config_label', 'exec_time_s', 'avg_time_per_round_s', 'speedup_vs_baseline']].copy()
summary_csv = TABLES_DIR / 'exp2_execution_summary.csv'
summary_md = TABLES_DIR / 'exp2_execution_summary.md'
summary_core.to_csv(summary_csv, index=False)
summary_md.write_text(to_markdown_no_deps(summary_core) + '\n', encoding='utf-8')

summary_details = summary[
    [
        'config_id',
        'config_label',
        'exec_time_s',
        'avg_time_per_round_s',
        'speedup_vs_baseline',
        'rounds_requested',
        'planned_rounds',
        'cached_rounds',
        'deduped_rounds',
        'parallelism_reported',
        'dedup_ratio',
        'cached_ratio',
        'mean_round_wall_time_s',
        'sum_round_wall_time_s',
        'max_completion_elapsed_s',
        'overlap_factor',
    ]
].copy()
summary_details_csv = TABLES_DIR / 'exp2_execution_summary_details.csv'
summary_details_md = TABLES_DIR / 'exp2_execution_summary_details.md'
summary_details.to_csv(summary_details_csv, index=False)
summary_details_md.write_text(to_markdown_no_deps(summary_details) + '\n', encoding='utf-8')

if not per_round_stats.empty:
    per_round_stats_csv = TABLES_DIR / 'exp2_per_round_walltime_summary.csv'
    per_round_stats.to_csv(per_round_stats_csv, index=False)

if not all_wall.empty:
    export_cols = [
        c
        for c in [
            'config_id',
            'config_label',
            'round',
            'wall_time_s',
            'completion_elapsed_s',
            'round_start_wallclock',
            'round_end_wallclock',
        ]
        if c in all_wall.columns
    ]
    all_wall_csv = TABLES_DIR / 'exp2_per_round_walltime_by_config.csv'
    all_wall[export_cols].to_csv(all_wall_csv, index=False)

print(f'wrote {summary_csv}')
print(f'wrote {summary_md}')
print(f'wrote {summary_details_csv}')
print(f'wrote {summary_details_md}')
if not per_round_stats.empty:
    print(f'wrote {per_round_stats_csv}')
if not all_wall.empty:
    print(f'wrote {all_wall_csv}')

summary_core


wrote /mnt/nas_drive/psklavos/MDM2026/fl-net-sim/experiments/exp2_parallelism_ablation/outputs/tables/exp2_execution_summary.csv
wrote /mnt/nas_drive/psklavos/MDM2026/fl-net-sim/experiments/exp2_parallelism_ablation/outputs/tables/exp2_execution_summary.md
wrote /mnt/nas_drive/psklavos/MDM2026/fl-net-sim/experiments/exp2_parallelism_ablation/outputs/tables/exp2_execution_summary_details.csv
wrote /mnt/nas_drive/psklavos/MDM2026/fl-net-sim/experiments/exp2_parallelism_ablation/outputs/tables/exp2_execution_summary_details.md
wrote /mnt/nas_drive/psklavos/MDM2026/fl-net-sim/experiments/exp2_parallelism_ablation/outputs/tables/exp2_per_round_walltime_summary.csv
wrote /mnt/nas_drive/psklavos/MDM2026/fl-net-sim/experiments/exp2_parallelism_ablation/outputs/tables/exp2_per_round_walltime_by_config.csv


,config_id,config_label,exec_time_s,avg_time_per_round_s,speedup_vs_baseline
0,baseline_ns3,Intra-process: Off | Inter-process: Off,668.919876,6.689199,1.000000
1,parallelism20_no_unison,Intra-process: Off | Inter-process: On,79.940318,0.799403,8.367741
2,parallelism20_unison_t2,Intra-process: On | Inter-process: On,46.354366,0.463544,14.430569
3,unison_auto_p1,Intra-process: On | Inter-process: Off,476.838256,4.768383,1.402823


In [10]:
fig1 = FIGURES_DIR / 'exp2_exec_time_bar.png'
fig2 = FIGURES_DIR / 'exp2_avg_exec_time_per_round_bar.png'
fig3 = FIGURES_DIR / 'exp2_wall_time_per_round_lines.png'

plt.figure(figsize=(11, 5))
plt.bar(summary['config_label'], summary['exec_time_s'])
plt.title('Total Execution Time by Parallelism Mode')
plt.ylabel('Execution Time (s)')
plt.xlabel('Parallelism Mode')
plt.xticks(rotation=15, ha='right')
plt.tight_layout()
plt.savefig(fig1, dpi=180)
plt.close()

plt.figure(figsize=(11, 5))
plt.bar(summary['config_label'], summary['avg_time_per_round_s'])
plt.title('Average Execution Time per Round')
plt.ylabel('Avg Exec Time / Round (s)')
plt.xlabel('Parallelism Mode')
plt.xticks(rotation=15, ha='right')
plt.tight_layout()
plt.savefig(fig2, dpi=180)
plt.close()

if not all_wall.empty and 'completion_elapsed_s' in all_wall.columns:
    plt.figure(figsize=(11, 5))
    group_col = 'config_label' if 'config_label' in all_wall.columns else 'config_id'
    requested_map = {}
    if {'config_id', 'rounds_requested'}.issubset(summary.columns):
        requested_map = (
            summary[['config_id', 'rounds_requested']]
            .dropna(subset=['config_id'])
            .drop_duplicates(subset=['config_id'])
            .set_index('config_id')['rounds_requested']
            .astype(int)
            .to_dict()
        )
    max_requested_rounds = 0
    for mode_label, group in all_wall.groupby(group_col):
        ordered = group.sort_values('completion_elapsed_s').reset_index(drop=True)
        completed_count = len(ordered)
        requested_rounds = completed_count
        if 'config_id' in ordered.columns and requested_map:
            config_id = ordered['config_id'].iloc[0]
            requested_rounds = int(requested_map.get(config_id, completed_count))
        requested_rounds = max(requested_rounds, completed_count, 1)
        if completed_count == 1:
            ordered['x_rounds_scaled'] = [requested_rounds]
        else:
            step = (requested_rounds - 1) / (completed_count - 1)
            ordered['x_rounds_scaled'] = [1 + i * step for i in range(completed_count)]
        max_requested_rounds = max(max_requested_rounds, requested_rounds)
        label = ordered['config_label'].iloc[0] if 'config_label' in ordered.columns else mode_label
        plt.plot(
            ordered['x_rounds_scaled'],
            ordered['completion_elapsed_s'],
            label=label,
            linewidth=1.4,
            marker='o',
            markersize=3,
        )
    plt.title('Completion Timeline by Parallelism Mode')
    plt.xlabel('Rounds')
    plt.ylabel('Wall-clock Time (s)')
    if max_requested_rounds > 0:
        tick_step = 10 if max_requested_rounds >= 20 else 1
        xticks = list(range(1, max_requested_rounds + 1, tick_step))
        if xticks[-1] != max_requested_rounds:
            xticks.append(max_requested_rounds)
        plt.xticks(xticks)
        plt.xlim(1, max_requested_rounds)
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(fig3, dpi=180)
    plt.close()

print(f'wrote {fig1}')
print(f'wrote {fig2}')
if not all_wall.empty:
    print(f'wrote {fig3}')


wrote /mnt/nas_drive/psklavos/MDM2026/fl-net-sim/experiments/exp2_parallelism_ablation/outputs/figures/exp2_exec_time_bar.png
wrote /mnt/nas_drive/psklavos/MDM2026/fl-net-sim/experiments/exp2_parallelism_ablation/outputs/figures/exp2_avg_exec_time_per_round_bar.png
wrote /mnt/nas_drive/psklavos/MDM2026/fl-net-sim/experiments/exp2_parallelism_ablation/outputs/figures/exp2_wall_time_per_round_lines.png
